# Fine-tuning QLoRA — HealthQA-BR (Llama-3-8B / Mistral-7B)

Este notebook treina um modelo biomédico em **português**, usando o dataset [Larxel/healthqa-br](https://huggingface.co/datasets/Larxel/healthqa-br) — questões de múltipla escolha de provas de residência/revalidação médica no Brasil (ex.: Revalida).

A resposta de treino é construída para **sempre** conter três elementos, exigidos pelo requisito de **segurança e validação**:

1. **Limites de atuação**: a resposta indica a alternativa correta e a justificativa, mas nunca representa uma prescrição/conduta definitiva sem validação humana.
2. **Explainability**: a resposta sempre cita a **fonte** da questão (`source` + `year` do dataset, ex.: "Revalida (2013)").
3. **Recomendação médica**: a resposta sempre recomenda o acompanhamento de um médico.

Esses três elementos são reforçados em duas camadas:
- **Na fonte de treino** (Seção 5): o texto-alvo do fine-tuning já inclui fonte + recomendação médica, para que o modelo aprenda esse comportamento como padrão.
- **No código** (Seção 10): guardrails verificam a resposta gerada e completam automaticamente qualquer um dos três elementos que estiver faltando, além de registrar tudo em log de auditoria.

**Antes de rodar:**
1. Menu `Ambiente de execução` → `Alterar tipo de ambiente de execução` → GPU → **T4**
2. Rode as células em ordem, de cima para baixo
3. O treino é controlado por `MAX_STEPS` (não por épocas), então o tempo é previsível

## 0. Atalho: usar o modelo já treinado (sem rodar o fine-tuning)

Se você só quer testar o modelo já publicado no Hugging Face, sem executar o pipeline de treinamento completo (seções 1 a 8), rode apenas a célula abaixo.

O modelo é baixado no formato GGUF Q4_K_M e roda localmente via llama-cpp-python. A célula detecta automaticamente se há GPU disponível, instala uma wheel pré-compilada (com ou sem CUDA, nunca compila do zero) e ajusta a inferência de acordo.

Repare que, mesmo sem aplicar nenhum guardrail de código aqui, a resposta já tende a trazer a fonte e a recomendação médica — isso é o comportamento aprendido no fine-tuning (Seção 5). A camada de guardrails (Seção 10) garante isso de forma determinística, para os casos em que o modelo não seguir o padrão.

In [ ]:
import subprocess
import sys

def _pip_install(args):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args]).returncode

# --- Validação de GPU ---
try:
    import torch
    GPU_DISPONIVEL = torch.cuda.is_available()
except ImportError:
    GPU_DISPONIVEL = False

if GPU_DISPONIVEL:
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")
else:
    print("Nenhuma GPU detectada. Rodando em CPU (mais lento).")

_pip_install(["huggingface_hub"])

# --- Instalação do llama-cpp-python ---
# Usa sempre wheels pré-compiladas (--only-binary=:all:), nunca compila do zero:
# compilar o backend CUDA do llama.cpp do zero pode levar mais de 1 hora no Colab.
# Se nenhuma wheel com CUDA for encontrada para a versão de CUDA do ambiente,
# cai para a wheel CPU padrão do PyPI (também pré-compilada, instalação rápida).
def _instalar_llama_cpp_cuda(tag: str) -> bool:
    return _pip_install([
        "--only-binary=:all:",
        "llama-cpp-python",
        "--extra-index-url", f"https://abetlen.github.io/llama-cpp-python/whl/{tag}",
    ]) == 0

if GPU_DISPONIVEL:
    tag_detectada = "cu" + (torch.version.cuda or "").replace(".", "")
    tags_candidatas = list(dict.fromkeys([tag_detectada, "cu124", "cu123", "cu122", "cu121"]))
    instalado_com_cuda = False
    for tag in tags_candidatas:
        if not tag or tag == "cu":
            continue
        print(f"Tentando wheel pré-compilada com CUDA ({tag})...")
        if _instalar_llama_cpp_cuda(tag):
            instalado_com_cuda = True
            print(f"llama-cpp-python instalado com suporte a CUDA ({tag}).")
            break
    if not instalado_com_cuda:
        print("Nenhuma wheel com CUDA disponível para este ambiente; usando versão CPU.")
        _pip_install(["--only-binary=:all:", "llama-cpp-python"])
        GPU_DISPONIVEL = False
else:
    _pip_install(["--only-binary=:all:", "llama-cpp-python"])

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

HF_REPO_ID = "jeferson2106/medico_ia_jeferson_br"  # ajuste para o repositório publicado na Seção 8
HF_GGUF_FILENAME = "llama-3-8b.Q4_K_M.gguf"

modelo_path = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_GGUF_FILENAME)

# -1 = offload de todas as camadas para GPU; 0 = roda inteiramente na CPU
N_GPU_LAYERS = -1 if GPU_DISPONIVEL else 0
llm_hf = Llama(model_path=modelo_path, n_ctx=2048, n_gpu_layers=N_GPU_LAYERS, verbose=False)
print(f"Modelo carregado em {'GPU' if GPU_DISPONIVEL else 'CPU'} (n_gpu_layers={N_GPU_LAYERS})")

ALPACA_PROMPT_HF = """Abaixo está uma questão médica de múltipla escolha (prova brasileira de residência/revalidação). Indique a alternativa correta, justifique brevemente, cite a fonte da questão e finalize sempre recomendando acompanhamento médico.

### Questão:
{}

### Resposta:
{}"""

def perguntar_ao_modelo_hf(questao: str, max_tokens: int = 250) -> str:
    prompt = ALPACA_PROMPT_HF.format(questao, "")
    saida = llm_hf(prompt, max_tokens=max_tokens, stop=["###"], echo=False)
    return saida["choices"][0]["text"].strip()

questao_teste = (
    "Paciente de 34 anos, sexo feminino, apresenta poliúria, polidipsia e perda de peso há 3 meses. "
    "Glicemia de jejum de 260 mg/dL em duas ocasiões.\n\n"
    "Qual é a conduta inicial mais adequada?\n\n"
    "A: orientação dietética isolada e reavaliação em 6 meses.\n"
    "B: iniciar insulinoterapia e encaminhar para acompanhamento endocrinológico.\n"
    "C: solicitar apenas hemoglobina glicada e aguardar resultado para decidir conduta.\n"
    "D: prescrever metformina sem necessidade de acompanhamento médico contínuo."
)

print(perguntar_ao_modelo_hf(questao_teste))

## 1. Instalação das dependências

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets
!pip install -q langchain langchain-community langchain-huggingface langgraph
!pip install -q "requests==2.32.4"

## 2. Montar o Google Drive (para salvar checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configurações gerais
Ajuste os parâmetros abaixo conforme necessidade.

In [ ]:
import os
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# Modelo base
MODEL_NAME = "unsloth/llama-3-8b-bnb-4bit"   # alternativa: "unsloth/mistral-7b-bnb-4bit"

# Dataset: questões de múltipla escolha de provas médicas brasileiras (PT-BR)
DATASET_NAME = "Larxel/healthqa-br"
DATASET_SPLIT = "train"
N_SAMPLES = None  # defina um número (ex: 2000) para limitar a quantidade de exemplos usados

# Comprimento máximo de sequência
MAX_SEQ_LENGTH = 1024

# Quantização 4-bit (obrigatório para caber no T4 com modelo 7B/8B)
LOAD_IN_4BIT = True

# LoRA
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

# Treino — controle direto por passos (não por épocas)
MAX_STEPS = 60          # ajuste conforme tempo disponível
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4    # batch efetivo = BATCH_SIZE * GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 2e-4
WARMUP_STEPS = 5
LOGGING_STEPS = 5
SAVE_STEPS = 20

# Saída
OUTPUT_DIR = "/content/drive/MyDrive/medllm_br_checkpoints"
FINAL_MODEL_DIR = "/content/drive/MyDrive/medllm_br_final"

# Frase fixa de recomendação médica -- exigida em toda resposta (treino e inferência)
RECOMENDACAO_MEDICA = (
    "Este conteúdo tem caráter educacional e não substitui a avaliação de um profissional de saúde. "
    "Consulte sempre um médico para confirmação diagnóstica e definição da conduta."
)

SEED = 3407

## 4. Carregar modelo base em 4-bit + aplicar LoRA

In [ ]:
print(f"Carregando modelo base: {MODEL_NAME}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # detecta automaticamente (fp16 no T4)
    load_in_4bit=LOAD_IN_4BIT,
)

print("Aplicando adaptadores LoRA")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

## 5. Carregar e formatar o dataset HealthQA-BR (múltipla escolha + fonte + recomendação médica)

O dataset tem os campos `question` (enunciado com as alternativas A-E embutidas), `answer` (letra correta), `source` (ex.: "Revalida") e `year`. Não existe um campo de "explicação" pronto, então a resposta de treino é **construída** a partir desses campos, incluindo sempre a fonte e a recomendação médica — para que o modelo aprenda esse padrão como comportamento default, e não apenas por instrução de prompt.

In [ ]:
import re

ALPACA_PROMPT = """Abaixo está uma questão médica de múltipla escolha (prova brasileira de residência/revalidação). Indique a alternativa correta, justifique brevemente, cite a fonte da questão e finalize sempre recomendando acompanhamento médico.

### Questão:
{}

### Resposta:
{}"""

def extrair_texto_alternativa(questao: str, letra: str) -> str:
    """Extrai o texto da alternativa correta (ex.: 'C: gota não tofácea; ...') a partir do enunciado."""
    padrao = rf"{letra}[:\)]\s*(.+?)(?=\n[A-E][:\)]|\Z)"
    m = re.search(padrao, questao, flags=re.DOTALL)
    if not m:
        return ""
    return " ".join(m.group(1).split())

def format_example(example, eos_token):
    questao = example["question"]
    letra_correta = str(example["answer"]).strip().upper()
    fonte = example.get("source") or "fonte não informada"
    ano = example.get("year")
    referencia = f"{fonte} ({ano})" if ano else fonte

    texto_alternativa = extrair_texto_alternativa(questao, letra_correta)
    justificativa = f"Alternativa correta: {letra_correta}"
    if texto_alternativa:
        justificativa += f" — {texto_alternativa}"

    resposta = (
        f"{justificativa}\n\n"
        f"Fonte: {referencia}.\n\n"
        f"{RECOMENDACAO_MEDICA}"
    )
    text = ALPACA_PROMPT.format(questao, resposta) + eos_token
    return {"text": text}


print(f"Carregando dataset: {DATASET_NAME}")
raw_dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT)

if N_SAMPLES is not None:
    raw_dataset = raw_dataset.select(range(min(N_SAMPLES, len(raw_dataset))))

print(f"Total de exemplos usados: {len(raw_dataset)}")

eos_token = tokenizer.eos_token
dataset = raw_dataset.map(
    lambda ex: format_example(ex, eos_token),
    remove_columns=raw_dataset.column_names,
)

# Conferir um exemplo formatado
print(dataset[0]["text"])

## 6. Treino
Controlado por `MAX_STEPS` — para exatamente quando atingir o número de passos definido acima.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported(),  # T4 não suporta bf16
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=LOGGING_STEPS,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=SEED,
    output_dir=OUTPUT_DIR,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    report_to="none",  # evita travar esperando login do W&B
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=training_args,
)

trainer_stats = trainer.train()
print(f"Treino concluído em {trainer_stats.metrics['train_runtime']:.1f}s")

## 7. Salvar modelo final (adaptadores LoRA)

In [ ]:
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"Modelo salvo em: {FINAL_MODEL_DIR}")

## 8. Exportar para GGUF e publicar no Hugging Face Hub

Necessário para rodar depois localmente via **Ollama**. Substitua `HF_REPO_ID` pelo seu usuário/nome-do-modelo e informe seu token do HF.

In [ ]:
from huggingface_hub import login

from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

HF_REPO_ID = "jeferson2106/medico_ia_jeferson_br"  # ajuste conforme necessário

model.push_to_hub_gguf(
    HF_REPO_ID,
    tokenizer,
    quantization_method="q4_k_m",
)

## 9. Teste rápido de inferência no próprio Colab

Útil para validar, antes de exportar, se a resposta já traz a fonte e a recomendação médica.

In [ ]:
FastLanguageModel.for_inference(model)

questao_teste = (
    "Paciente de 34 anos, sexo feminino, apresenta poliúria, polidipsia e perda de peso há 3 meses. "
    "Glicemia de jejum de 260 mg/dL em duas ocasiões.\n\n"
    "Qual é a conduta inicial mais adequada?\n\n"
    "A: orientação dietética isolada e reavaliação em 6 meses.\n"
    "B: iniciar insulinoterapia e encaminhar para acompanhamento endocrinológico.\n"
    "C: solicitar apenas hemoglobina glicada e aguardar resultado para decidir conduta.\n"
    "D: prescrever metformina sem necessidade de acompanhamento médico contínuo."
)

prompt = ALPACA_PROMPT.format(questao_teste, "")
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

## 10. Integração com LangChain/LangGraph e guardrails reforçados (segurança e validação)

Conecta o modelo fine-tuned a um pipeline LangChain que consulta um prontuário simulado, contextualiza a resposta e organiza o atendimento com LangGraph. A camada de guardrails abaixo **garante em código**, de forma determinística, os três pontos do requisito de segurança e validação — mesmo nos casos em que o modelo não siga o padrão aprendido no treino:

1. `garantir_limites_de_atuacao`: sinaliza quando a resposta sugere uma conduta/medicação direta, avisando que exige validação humana.
2. `garantir_fonte`: se a resposta não citar "Fonte:", uma é adicionada automaticamente (com base no dado consultado).
3. `garantir_recomendacao_medica`: se a resposta não recomendar acompanhamento médico, a frase padrão é adicionada.

Cada etapa é registrada em log de auditoria.

In [ ]:
import json
import logging
from datetime import datetime, UTC

# Log de auditoria: cada evento do fluxo é registrado com timestamp para rastreamento
LOG_PATH = "/content/drive/MyDrive/medllm_br_checkpoints/assistente_auditoria.log"

logging.basicConfig(
    filename=LOG_PATH,
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
)
logger_auditoria = logging.getLogger("assistente_medico_br")

def registrar_log(etapa: str, dados: dict) -> dict:
    """Registra um evento do fluxo para auditoria e rastreamento (requisito de segurança)."""
    registro = {"timestamp": datetime.now(UTC).isoformat(), "etapa": etapa, **dados}
    logger_auditoria.info(json.dumps(registro, ensure_ascii=False, default=str))
    return registro

In [ ]:
# Termos que indicam prescrição/conduta direta -- exigem validação humana explícita
TERMOS_PRESCRICAO = [
    "take ", "administer ", "i prescribe ", "prescription for ", "dose of ", "apply ",
    "tome ", "administre ", "prescrevo ", "receita de ", "dose de ", "aplique ",
    "inicie o uso de", "iniciar o uso de",
]

def contem_prescricao(resposta: str) -> bool:
    resposta_lower = resposta.lower()
    return any(termo in resposta_lower for termo in TERMOS_PRESCRICAO)

def garantir_limites_de_atuacao(resposta: str) -> str:
    """
    Limite de atuação do assistente: nunca prescrever/decidir diretamente sem validação humana.
    Filtro baseado em palavras-chave; em produção, substituir por um classificador dedicado.
    """
    if contem_prescricao(resposta):
        resposta += (
            "\n\nAVISO DE SEGURANÇA: esta resposta sugere uma conduta ou medicação. "
            "O assistente NÃO substitui o julgamento clínico -- toda prescrição exige "
            "validação e assinatura de um médico responsável antes de ser aplicada ao paciente."
        )
    return resposta

def garantir_fonte(resposta: str, referencia_padrao: str) -> str:
    """Explainability: garante que a resposta cite a fonte da informação usada."""
    if "fonte:" not in resposta.lower() and "source:" not in resposta.lower():
        resposta += f"\n\nFonte: {referencia_padrao}."
    return resposta

def garantir_recomendacao_medica(resposta: str) -> str:
    """Garante que a resposta sempre recomende acompanhamento médico."""
    marcadores = ["consulte", "acompanhamento médico", "avaliação médica", "profissional de saúde"]
    resposta_lower = resposta.lower()
    if not any(marcador in resposta_lower for marcador in marcadores):
        resposta += f"\n\n{RECOMENDACAO_MEDICA}"
    return resposta

def aplicar_guardrails(resposta: str, referencia_padrao: str = "conhecimento clínico geral") -> str:
    """Aplica, em sequência, os três requisitos de segurança e validação."""
    prescricao_detectada = contem_prescricao(resposta)
    resposta = garantir_limites_de_atuacao(resposta)
    resposta = garantir_fonte(resposta, referencia_padrao)
    resposta = garantir_recomendacao_medica(resposta)
    registrar_log("guardrail_aplicado", {"prescricao_detectada": prescricao_detectada})
    return resposta

In [ ]:
import pandas as pd
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from transformers import pipeline as hf_pipeline

# --- Wrapper do modelo fine-tuned como LLM do LangChain ---
FastLanguageModel.for_inference(model)

gerador = hf_pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    do_sample=False,
    temperature=0.1,
    return_full_text=False,
)
llm = HuggingFacePipeline(pipeline=gerador)

# --- Base de dados estruturada simulada (prontuários eletrônicos) ---
# Em produção, isso viria do sistema hospitalar (HL7/FHIR, banco relacional, etc.).
# Aqui usamos dados sintéticos para demonstrar o fluxo, sem expor dados reais de pacientes.
prontuarios_df = pd.DataFrame([
    {
        "paciente_id": "P001",
        "idade": 62,
        "condicoes": "Diabetes tipo 2, Hipertensão",
        "exames_pendentes": ["Hemoglobina glicada (HbA1c)", "Perfil lipídico"],
        "exames_realizados": {"Glicemia de jejum": "182 mg/dL", "Creatinina": "1.1 mg/dL"},
        "medicacoes_atuais": ["Metformina 850mg 2x/dia"],
    },
    {
        "paciente_id": "P002",
        "idade": 45,
        "condicoes": "Asma",
        "exames_pendentes": [],
        "exames_realizados": {"Espirometria": "Normal"},
        "medicacoes_atuais": ["Salbutamol conforme necessidade"],
    },
])

def consultar_prontuario(paciente_id: str) -> dict:
    """Tool de consulta ao prontuário estruturado do paciente (requisito: consultas em base estruturada)."""
    linha = prontuarios_df[prontuarios_df["paciente_id"] == paciente_id]
    return linha.iloc[0].to_dict() if not linha.empty else {}

# --- Prompt contextualizado com dados do paciente + exigência de explainability ---
ASSISTENTE_PROMPT = PromptTemplate.from_template(
    "Você é um assistente médico virtual do hospital. Responda com base apenas no "
    "protocolo interno e nos dados do paciente abaixo. Ao final, cite a fonte da "
    "informação usada (protocolo interno, prontuário do paciente ou conhecimento clínico geral) "
    "e sempre recomende o acompanhamento de um médico.\n\n"
    "### Protocolo / contexto clínico:\n{contexto}\n\n"
    "### Dados atuais do paciente:\n{dados_paciente}\n\n"
    "### Pergunta do médico:\n{pergunta}\n\n"
    "### Resposta (finalize com \'Fonte:\' e uma recomendação de acompanhamento médico):\n"
)
def responder_com_contexto(pergunta: str, paciente_id: str, contexto_clinico: str = "Nenhum protocolo adicional fornecido.") -> dict:
    """Pipeline LangChain: consulta o prontuário, contextualiza o prompt e aplica os guardrails de segurança."""
    dados_paciente = consultar_prontuario(paciente_id)
    referencia_padrao = "prontuário do paciente" if dados_paciente else "conhecimento clínico geral"
    registrar_log("consulta_prontuario", {"paciente_id": paciente_id, "encontrado": bool(dados_paciente)})

    prompt = ASSISTENTE_PROMPT.format(
        contexto=contexto_clinico,
        dados_paciente=json.dumps(dados_paciente, ensure_ascii=False, default=str),
        pergunta=pergunta,
    )
    resposta_bruta = llm.invoke(prompt)
    resposta_validada = aplicar_guardrails(resposta_bruta, referencia_padrao=referencia_padrao)

    registrar_log("resposta_gerada", {"paciente_id": paciente_id, "pergunta": pergunta})
    return {"resposta": resposta_validada, "dados_paciente": dados_paciente}

In [ ]:
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, START, END

class EstadoAtendimento(TypedDict):
    paciente_id: str
    pergunta: str
    dados_paciente: dict
    exames_pendentes: List[str]
    resposta_llm: Optional[str]
    alertas: List[str]

def no_receber_paciente(estado: EstadoAtendimento) -> dict:
    dados = consultar_prontuario(estado["paciente_id"])
    registrar_log("no_receber_paciente", {"paciente_id": estado["paciente_id"]})
    return {"dados_paciente": dados}

def no_verificar_exames(estado: EstadoAtendimento) -> dict:
    pendentes = estado["dados_paciente"].get("exames_pendentes", [])
    registrar_log("no_verificar_exames", {"pendentes": pendentes})
    return {"exames_pendentes": pendentes}

def no_emitir_alertas(estado: EstadoAtendimento) -> dict:
    alertas = [f"Exame pendente: {exame}" for exame in estado["exames_pendentes"]]
    registrar_log("no_emitir_alertas", {"alertas": alertas})
    return {"alertas": alertas}

def no_sugerir_conduta(estado: EstadoAtendimento) -> dict:
    resultado = responder_com_contexto(estado["pergunta"], estado["paciente_id"])
    registrar_log("no_sugerir_conduta", {"paciente_id": estado["paciente_id"]})
    return {"resposta_llm": resultado["resposta"]}

def rota_apos_exames(estado: EstadoAtendimento) -> str:
    """Fluxo de decisão: paciente com exames pendentes recebe alerta antes da sugestão de conduta."""
    return "com_pendencia" if estado["exames_pendentes"] else "sem_pendencia"

grafo = StateGraph(EstadoAtendimento)
grafo.add_node("receber_paciente", no_receber_paciente)
grafo.add_node("verificar_exames", no_verificar_exames)
grafo.add_node("emitir_alertas", no_emitir_alertas)
grafo.add_node("sugerir_conduta", no_sugerir_conduta)

grafo.add_edge(START, "receber_paciente")
grafo.add_edge("receber_paciente", "verificar_exames")
grafo.add_conditional_edges(
    "verificar_exames",
    rota_apos_exames,
    {"com_pendencia": "emitir_alertas", "sem_pendencia": "sugerir_conduta"},
)
grafo.add_edge("emitir_alertas", "sugerir_conduta")
grafo.add_edge("sugerir_conduta", END)

fluxo_assistente = grafo.compile()
print("Grafo LangGraph compilado com sucesso.")

In [ ]:
estado_inicial = {
    "paciente_id": "P001",
    "pergunta": "Qual a conduta inicial mais adequada para o controle glicêmico deste paciente?",
    "dados_paciente": {},
    "exames_pendentes": [],
    "resposta_llm": None,
    "alertas": [],
}

resultado_final = fluxo_assistente.invoke(estado_inicial)

print("Alertas emitidos:")
for alerta in resultado_final["alertas"]:
    print(f" - {alerta}")

print("\nResposta do assistente médico:\n")
print(resultado_final["resposta_llm"])

print(f"\nLog de auditoria salvo em: {LOG_PATH}")

## 11. Diagrama do fluxo (para o relatório técnico)

Gera a representação do grafo LangGraph em Mermaid (diagrama do fluxo LangChain/LangGraph).

In [ ]:
# Gera a representação Mermaid do grafo -- útil para colar no relatório técnico
print(fluxo_assistente.get_graph().draw_mermaid())